# Źródła prognoz pogody — przegląd

Notebook pokazuje **trzy sposoby** pozyskiwania prognoz pogody:

| # | Źródło | Co to jest | Wymaga auth |
|---|--------|-----------|-------------|
| 1 | **Open-Meteo API** | REST API — najłatwiejszy sposób na prognozy (600 calli/min, 5000 calli/h) | Nie |
| 2 | **Open-Meteo S3 bucket** | Surowe dane modeli na S3 (to repo z nich pobiera) | Nie |
| 3 | **ECMWF Open Data** | Oficjalne pliki GRIB z ECMWF przez S3 | Nie |

Dla każdego źródła pokazujemy pobranie **temperatury 2m** — zarówno **najnowszej prognozy**, jak i **historycznej**

---
## 1 · Open-Meteo API

Najprostszy punkt wejścia — darmowe API (z limitami) z cache'owaniem, wieloma modelami i bez rejestracji.  
Dokumentacja: https://open-meteo.com/en/docs

In [3]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [4]:
# Najnowsza prognoza dla Berlina
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 52.52,
    "longitude": 13.41,
    "hourly": "temperature_2m",
}
response = openmeteo.weather_api(url, params=params)[0]

print(f"Współrzędne : {response.Latitude():.2f}°N {response.Longitude():.2f}°E")
print(f"Wysokość    : {response.Elevation()} m n.p.m.")

hourly = response.Hourly()
df_latest = pd.DataFrame({
    "time": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ),
    "t2m": hourly.Variables(0).ValuesAsNumpy(),
})
df_latest.head(10)

Współrzędne : 52.52°N 13.42°E
Wysokość    : 38.0 m n.p.m.


,time,t2m
0,2026-03-28 00:00:00+00:00,2.6955
1,2026-03-28 01:00:00+00:00,2.3455
2,2026-03-28 02:00:00+00:00,1.7955
3,2026-03-28 03:00:00+00:00,0.9455
4,2026-03-28 04:00:00+00:00,0.8955
5,2026-03-28 05:00:00+00:00,0.7955
6,2026-03-28 06:00:00+00:00,0.8955
7,2026-03-28 07:00:00+00:00,1.6955
8,2026-03-28 08:00:00+00:00,3.4955
9,2026-03-28 09:00:00+00:00,7.0955


In [5]:
# Historyczne prognozy — previous-runs API
# Osobny endpoint: dla danego momentu X dostajemy prognozy wydane
# 5/4/3/2/1 dni przed X, co pozwala porównać jak prognoza się zmieniała.
url_hist = "https://previous-runs-api.open-meteo.com/v1/forecast"
params_hist = {
    "latitude": 52.52,
    "longitude": 13.41,
    "hourly": [
        "temperature_2m",
        "temperature_2m_previous_day1",
        "temperature_2m_previous_day5",
    ],
}
resp_hist = openmeteo.weather_api(url_hist, params=params_hist)[0]

h = resp_hist.Hourly()
df_hist = pd.DataFrame({
    "time":     pd.date_range(
        start=pd.to_datetime(h.Time(), unit="s", utc=True),
        end=pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=h.Interval()),
        inclusive="left",
    ),
    "t2m":      h.Variables(0).ValuesAsNumpy(),
    "t2m_-1d":  h.Variables(1).ValuesAsNumpy(),
    "t2m_-5d":  h.Variables(2).ValuesAsNumpy(),
})
df_hist.head(10)

,time,t2m,t2m_-1d,t2m_-5d
0,2026-03-28 00:00:00+00:00,2.6955,3.389,2.1195
1,2026-03-28 01:00:00+00:00,2.3455,3.489,1.6695
2,2026-03-28 02:00:00+00:00,1.7955,3.639,1.1695
3,2026-03-28 03:00:00+00:00,0.9455,3.789,0.7195
4,2026-03-28 04:00:00+00:00,0.8955,3.789,0.2195
5,2026-03-28 05:00:00+00:00,0.7955,3.789,-0.1805
6,2026-03-28 06:00:00+00:00,0.8955,3.889,0.1195
7,2026-03-28 07:00:00+00:00,1.6955,4.189,1.6695
8,2026-03-28 08:00:00+00:00,3.4955,4.589,3.9695
9,2026-03-28 09:00:00+00:00,7.0955,5.089,6.0695


---
## 2 · Open-Meteo S3 Bucket (surowe siatki modeli)

Open-Meteo mirroruje modele na **publicznym buckecie S3** (`openmeteo.s3.amazonaws.com`).  
To właśnie z tego źródła api OM korzysta pod spodem

### Dostępne modele dla `temperature_2m` 

| Model | Organizacja | Przybliżony horyzont |
|-------|-------------|----------------------|
| `ecmwf_ifs025` | ECMWF | 144 h |
| `ecmwf_aifs025_single` | ECMWF | 168 h |
| `ncep_gfs013` | NCEP | 168 h |
| `ncep_gfs_graphcast025` | NCEP | 168 h |
| `ncep_aigfs025` | NCEP | 168 h |
| `dwd_icon` | DWD | 120 h |
| `meteofrance_arpege_world025` | Météo-France | 102 h |
| `ukmo_global_deterministic_10km` | UK Met Office | 60 h |
| `kma_gdps` | Korea Met | 168 h |

najpopularniejsze według

https://openmeteo.substack.com/p/best-weather-models-in-one-open-source

In [6]:
import requests
import json
import numpy as np

S3_BASE = "https://openmeteo.s3.amazonaws.com"

def om_bucket_meta(model: str) -> dict:
    """Pobierz metadane chunków dla modelu z bucketu OM."""
    url = f"{S3_BASE}/data/{model}/static/meta.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def om_run_url(model: str, run: str, variable: str) -> str:
    """Zbuduj URL do pliku konkretnego biegu modelu (dane prognostyczne).
    Format run: '2025/03/25/1200Z'"""
    return f"{S3_BASE}/data_run/{model}/{run}/{variable}.om"

In [7]:
# Sprawdzamy jak wyglądają metadane dla ECMWF IFS
meta = om_bucket_meta("ecmwf_ifs025")
print(json.dumps({k: v for k, v in meta.items() if k != "chunks"}, indent=2))

{
  "chunk_time_length": 104,
  "crs_wkt": "GEOGCRS[\"WGS 84\",\n    DATUM[\"World Geodetic System 1984\",\n        ELLIPSOID[\"WGS 84\",6378137,298.257223563]],\n    CS[ellipsoidal,2],\n        AXIS[\"latitude\",north],\n        AXIS[\"longitude\",east],\n        ANGLEUNIT[\"degree\",0.0174532925199433]\n    USAGE[\n        SCOPE[\"grid\"],\n        BBOX[-90.0,-180.0,90.0,179.75]]]",
  "data_end_time": 1775206800,
  "last_run_availability_time": 1774704035,
  "last_run_initialisation_time": 1774677600,
  "last_run_modification_time": 1774704035,
  "temporal_resolution_seconds": 10800,
  "update_interval_seconds": 21600
}


In [8]:
# Sprawdzenie dostępności historycznego biegu (2025-03-25 12:00Z)
hist_url = om_run_url("ecmwf_ifs025", "2026/03/25/1200Z", "temperature_2m")
r = requests.head(hist_url, timeout=10)
print(f"Historyczny URL  : {hist_url}")
print(f"Status           : {r.status_code} ({'dostępny' if r.ok else 'nie znaleziono'})")

# Sprawdzenie najnowszego biegu (dzisiejszy 00Z)
from datetime import datetime, UTC
now = datetime.now(UTC)
latest_run = now.strftime("%Y/%m/%d") + "/0000Z"
latest_url = om_run_url("ecmwf_ifs025", latest_run, "temperature_2m")
r2 = requests.head(latest_url, timeout=10)
print(f"\nNajnowszy URL    : {latest_url}")
print(f"Status           : {r2.status_code} ({'dostępny' if r2.ok else 'nie znaleziono'})")

Historyczny URL  : https://openmeteo.s3.amazonaws.com/data_run/ecmwf_ifs025/2026/03/25/1200Z/temperature_2m.om
Status           : 200 (dostępny)

Najnowszy URL    : https://openmeteo.s3.amazonaws.com/data_run/ecmwf_ifs025/2026/03/28/0000Z/temperature_2m.om
Status           : 200 (dostępny)


---
## 3 · ECMWF Open Data (HTTPS)

ECMWF publikuje prognozy operacyjne jako **pliki GRIB** na publicznym serwerze.  
Cięższe w użyciu, kompletne i oryginalne.

- Endpoint: `https://data.ecmwf.int/forecasts/`
- Struktura: `{data}/{HHz}/ifs/0p25/oper/{data}000000-{step}h-oper-fc.grib2`
- Dostępne prognozy historyczne
- Każdy plik GRIB zawiera **wszystkie zmienne** dla danego kroku — pliki ~130 MB
- Dokumentacja: https://www.ecmwf.int/en/forecasts/datasets/open-data

In [9]:
import re

ECMWF_BASE = "https://data.ecmwf.int/forecasts"

def ecmwf_list_runs(date: str) -> list[str]:
    """Listuj dostępne biegi dla danej daty (YYYYMMDD)."""
    r = requests.get(f"{ECMWF_BASE}/{date}/", timeout=10)
    r.raise_for_status()
    return re.findall(r'href="[^"]*?/(\d{2}z)/"', r.text)

def ecmwf_list_products(date: str, run: str) -> list[str]:
    """Listuj dostępne produkty (oper, scda, enfo, ...) dla biegu."""
    url = f"{ECMWF_BASE}/{date}/{run}/ifs/0p25/"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    return re.findall(r'href="[^"]*?/(\w+)/"', r.text)

def ecmwf_list_steps(date: str, run: str, product: str = "oper") -> list[str]:
    """Listuj dostępne kroki czasowe (pliki GRIB) dla biegu.
    00z/12z mają 'oper', 06z/18z mają 'scda'."""
    url = f"{ECMWF_BASE}/{date}/{run}/ifs/0p25/{product}/"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    files = re.findall(r'href="[^"]*?(\d+h-\w+-fc\.grib2)"', r.text)
    return sorted(files, key=lambda f: int(f.split("h-")[0]))

def ecmwf_grib_url(date: str, run: str, step_file: str, product: str = "oper") -> str:
    """Zbuduj pełny URL do pliku GRIB."""
    return f"{ECMWF_BASE}/{date}/{run}/ifs/0p25/{product}/{step_file}"

In [10]:
# Jakie daty i runy są dostępne?
today = datetime.now(UTC).strftime("%Y%m%d")

runs = ecmwf_list_runs(today)
print(f"Dostępne runy na {today}: {runs}")

# Produkty dla najnowszego biegu
latest_run = runs[-1]
products = ecmwf_list_products(today, latest_run)
print(f"Produkty dla {latest_run}: {products}")

# Wybierz właściwy produkt: 00z/12z → oper, 06z/18z → scda
product = "oper" if "oper" in products else "scda"

# Kroki czasowe
steps = ecmwf_list_steps(today, latest_run, product)
print(f"\nBieg {latest_run} ({product}) — dostępne kroki ({len(steps)}):")
print(", ".join(s.split("h-")[0] + "h" for s in steps[:15]), "...")

Dostępne runy na 20260328: ['00z', '06z']
Produkty dla 06z: ['forecasts', '20260328', '06z', 'ifs', '0p25', 'enfo', 'scda', 'scwv', 'waef', 'ifs']

Bieg 06z (scda) — dostępne kroki (49):
0h, 3h, 6h, 9h, 12h, 15h, 18h, 21h, 24h, 27h, 30h, 33h, 36h, 39h, 42h ...


In [11]:
url_6h = ecmwf_grib_url(today, latest_run, steps[1], product)
r = requests.head(url_6h, timeout=10)
print(f"URL    : {url_6h}")

URL    : https://data.ecmwf.int/forecasts/20260328/06z/ifs/0p25/scda/3h-scda-fc.grib2


---
## Podsumowanie

| Źródło | Zalety | Wady |
|--------|--------|------|
| **Open-Meteo API** | Zero konfiguracji, zapytania punktowe, wiele modeli | Limity zapytań, brak surowych siatek |
| **Open-Meteo S3** | Pełne siatki, publiczny, 9+ modeli | Własny format `.om` |
| **ECMWF Open Data** | Autorytatywne GRIB, globalne | Duże pliki, parsowanie GRIB |